# NB01 — Data Prep

In [1]:
import os, json, random, warnings, torch
import numpy as np
from dataclasses import dataclass, asdict, field
from typing import List, Dict, Tuple, Optional
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

MODELS = {
    "smollm3": "HuggingFaceTB/SmolLM3-3B",
    "qwen25":  "Qwen/Qwen2.5-3B-Instruct",
}
PRIMARY_MODEL = "smollm3"

SAE_EPOCHS       = 25
SAE_EXPANSION    = 4
SAE_L1           = 1e-3
SAE_LR           = 1e-4
SAE_BATCH        = 64
ACT_SEQ_LEN      = 256

TRAIN_SIZE = 8_000
VAL_SIZE   = 2_000
TEST_SIZE  = 2_000
SEED       = 42

DATASET_SELECTIONS = {
    "medical": {"source": "medmcqa",
                "drift": "Factual precision drift — sharpen truth-conditional reasoning"},
    "code":    {"source": "iamtarun/python_code_instructions_18k_alpaca",
                "drift": "Structural abstraction drift — syntax-level attention rotation"},
    "finance": {"source": "zeroshot/twitter-financial-news-sentiment",
                "drift": "Risk-sensitivity drift — compress uncertainty representations"},
}

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")
print(f"Primary model: {MODELS[PRIMARY_MODEL]}")

PyTorch: 2.12.0.dev20260326+cu128
CUDA: True — NVIDIA GeForce RTX 5080 Laptop GPU
VRAM: 17.1 GB
Primary model: HuggingFaceTB/SmolLM3-3B


C:\Users\vihaa\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from datasets import load_dataset, Dataset, DatasetDict

def _split(ds):
    ds = ds.shuffle(seed=SEED)
    n = len(ds)
    total_needed = TRAIN_SIZE + VAL_SIZE + TEST_SIZE
    if n >= total_needed:
        tr, va, te = TRAIN_SIZE, VAL_SIZE, TEST_SIZE
    else:
        tr = int(n * 0.67)
        va = int(n * 0.165)
        te = n - tr - va
        print(f"    Small dataset ({n} rows) → {tr}/{va}/{te} split")
    assert tr + va + te <= n, f"Split exceeds dataset size: {tr}+{va}+{te} > {n}"
    return DatasetDict({"train": ds.select(range(tr)),
                        "validation": ds.select(range(tr, tr + va)),
                        "test": ds.select(range(tr + va, tr + va + te))})

def _save(dd, name):
    out = f"data/{name}"
    os.makedirs(out, exist_ok=True)
    for split, ds in dd.items():
        ds.save_to_disk(f"{out}/{split}")
    tr_ids = set(range(len(dd["train"])))
    va_ids = set(range(len(dd["train"]), len(dd["train"]) + len(dd["validation"])))
    assert len(tr_ids & va_ids) == 0, "Overlap detected!"
    print(f"  {name:10s}  train={len(dd['train']):,}  val={len(dd['validation']):,}  test={len(dd['test']):,}")

print("=== Medical (medmcqa — 182k rows) ===")
raw = load_dataset("medmcqa", split="train")
OPT_MAP = {0: "opa", 1: "opb", 2: "opc", 3: "opd"}
def fmt_med(ex):
    opts = "\n".join([f"  {chr(65+i)}) {ex[k]}" for i, k in enumerate(["opa","opb","opc","opd"]) if ex.get(k)])
    correct = ex.get(OPT_MAP.get(ex.get("cop", -1), ""), "")
    exp = ex.get("exp", "")
    answer = f"{correct}" + (f"\n{exp}" if exp else "")
    return {"instruction": "Answer this medical question.",
            "input": f"{ex['question']}\n{opts}",
            "output": answer, "domain": "medical"}
_save(_split(raw.map(fmt_med, remove_columns=raw.column_names)), "medical")

print("=== Code (python_code_instructions — 18k rows) ===")
raw = load_dataset("iamtarun/python_code_instructions_18k_alpaca", split="train")
def fmt_code(ex):
    return {"instruction": str(ex.get("instruction", "")),
            "input": str(ex.get("input", "")),
            "output": str(ex.get("output", "")), "domain": "code"}
_save(_split(raw.map(fmt_code, remove_columns=raw.column_names)), "code")

print("=== Finance (twitter-financial-news — 9.5k rows) ===")
raw = load_dataset("zeroshot/twitter-financial-news-sentiment", split="train")
LABEL_MAP = {0: "Bearish", 1: "Bullish", 2: "Neutral"}
def fmt_fin(ex):
    return {"instruction": "Classify this financial news sentiment as Bearish, Bullish, or Neutral.",
            "input": str(ex.get("text", "")),
            "output": LABEL_MAP.get(ex.get("label", 2), "Neutral"), "domain": "finance"}
_save(_split(raw.map(fmt_fin, remove_columns=raw.column_names)), "finance")

os.makedirs("data", exist_ok=True)
json.dump(DATASET_SELECTIONS, open("data/manifest.json", "w"), indent=2)
print("\nAll datasets ready — no overlap, no leakage.")

=== Medical (medmcqa — 182k rows) ===


Saving the dataset (1/1 shards): 100%|██████████| 2000/2000 [00:00<00:00, 69656.05 examples/s]


  medical     train=8,000  val=2,000  test=2,000
=== Code (python_code_instructions — 18k rows) ===


Saving the dataset (1/1 shards): 100%|██████████| 2000/2000 [00:00<00:00, 137044.13 examples/s]


  code        train=8,000  val=2,000  test=2,000
=== Finance (twitter-financial-news — 9.5k rows) ===
    Small dataset (9543 rows) → 6393/1574/1576 split


Saving the dataset (1/1 shards): 100%|██████████| 1576/1576 [00:00<00:00, 151322.55 examples/s]

  finance     train=6,393  val=1,574  test=1,576

All datasets ready — no overlap, no leakage.
